# Classification of Inflammatory Bowel Disease Using Microbial Taxonomic Abundance

**Final machine learning pipeline**

This  notebook builds a reproducible supervised learning workflow for classifying inflammatory bowel disease (IBD) status from microbial taxonomic abundance data.

**Final dataset direction:** HMP2 / IBDMDB metagenomic taxonomic profiles with public metadata.

**Primary target:** binary classification, `IBD` (`CD` or `UC`) vs `nonIBD`.

**Clinical evaluation priority:** recall/sensitivity for the IBD class, while also reporting specificity so false positives are visible.

Run this notebook top-to-bottom in Google Colab. The data download is automatic.

## 1. Setup and Imports

This cell imports the required Python libraries and creates output folders. It also installs missing packages if the runtime does not already provide them.

In [ ]:
#@title 1.1 Setup and imports
import sys
import subprocess
import warnings
from pathlib import Path

def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

for import_name, pip_name in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("sklearn", "scikit-learn"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
]:
    ensure_package(import_name, pip_name)

import urllib.request
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

BASE_DIR = Path("/content/ibd_microbiome_project") if IN_COLAB else Path(".")
DATA_DIR = BASE_DIR / "data" / "raw"
OUTPUT_DIR = BASE_DIR / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Running in Colab: {IN_COLAB}")
print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

## 2. Dataset Source

The final pipeline uses HMP2 / IBDMDB public metagenomic taxonomic profiles and metadata.

- Metadata: `hmp2_metadata_2018-08-20.csv`
- Taxonomic table: `taxonomic_profiles_3.tsv.gz`
- Diagnosis labels in metadata: `CD`, `UC`, `nonIBD`
- Binary target used here: `CD` and `UC` become `IBD`; `nonIBD` remains `nonIBD`

This dataset is stronger than the original milestone 1 MLRepo direction because it has more samples and subject IDs, allowing subject-aware leakage prevention.

In [ ]:
#@title 2.1 Download HMP2 / IBDMDB files
HMP2_METADATA_URL = "https://g-227ca.190ebd.75bc.data.globus.org/ibdmdb/metadata/hmp2_metadata_2018-08-20.csv"
HMP2_TAXONOMY_URL = "https://g-227ca.190ebd.75bc.data.globus.org/ibdmdb/products/HMP2/MGX/2018-05-04/taxonomic_profiles_3.tsv.gz"

metadata_path = DATA_DIR / "hmp2_metadata_2018-08-20.csv"
taxonomy_path = DATA_DIR / "taxonomic_profiles_3.tsv.gz"

def download_if_needed(url, destination):
    destination = Path(destination)
    if destination.exists() and destination.stat().st_size > 0:
        print(f"Already downloaded: {destination.name} ({destination.stat().st_size:,} bytes)")
        return destination

    print(f"Downloading {destination.name} ...")
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=120) as response, destination.open("wb") as output_file:
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            output_file.write(chunk)
    print(f"Saved: {destination.name} ({destination.stat().st_size:,} bytes)")
    return destination

metadata_path = download_if_needed(HMP2_METADATA_URL, metadata_path)
taxonomy_path = download_if_needed(HMP2_TAXONOMY_URL, taxonomy_path)

## 3. Preprocessing and Leakage-Safe Split

All core preprocessing is handled in this cell:

- Load metadata and metagenomic taxonomic profiles.
- Keep metagenomics samples only.
- Convert `CD` and `UC` to binary `IBD`.
- Use genus-level taxa so the feature space is interpretable and course-aligned.
- Merge taxonomic abundance rows with metadata by sample ID.
- Split by participant ID, not random rows, to prevent leakage from repeated longitudinal samples.
- Fit feature filtering only from the training data.
- Apply `log1p` transformation to reduce sparse abundance skew.

In [ ]:
#@title 3.1 Preprocessing functions and execution
def load_hmp2_metadata(path):
    metadata = pd.read_csv(path, low_memory=False)
    metadata = metadata[metadata["data_type"].eq("metagenomics")].copy()
    metadata["IBD_Status"] = metadata["diagnosis"].map({"nonIBD": 0, "CD": 1, "UC": 1})
    metadata = metadata.dropna(subset=["External ID", "Participant ID", "diagnosis", "IBD_Status"])
    metadata["IBD_Status"] = metadata["IBD_Status"].astype(int)
    return metadata

def load_genus_abundance_matrix(path):
    taxonomy = pd.read_csv(path, sep="\t", compression="gzip")
    feature_col = taxonomy.columns[0]

    genus_mask = (
        taxonomy[feature_col].astype(str).str.contains(r"\|g__", regex=True)
        & ~taxonomy[feature_col].astype(str).str.contains(r"\|s__", regex=True)
        & ~taxonomy[feature_col].astype(str).str.endswith("g__")
    )
    genus = taxonomy[genus_mask].copy()

    X = genus.set_index(feature_col).T
    X.columns = [re.sub(r"^.*\|g__", "", str(col)) for col in X.columns]
    X.index = X.index.to_series().str.replace(r"_profile$", "", regex=True)
    X = X.apply(pd.to_numeric, errors="coerce").fillna(0.0)
    X = X.T.groupby(level=0).sum().T
    X = X.groupby(level=0).median()
    return X

def merge_metadata_and_features(metadata, X):
    merged = metadata.set_index("External ID").join(X, how="inner")
    merged = merged.reset_index().rename(columns={"index": "Sample ID"})
    return merged

def subject_aware_train_test_split(X, y, groups, test_size=0.20, random_state=42):
    for seed in range(random_state, random_state + 500):
        splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
        train_idx, test_idx = next(splitter.split(X, y, groups))
        if y.iloc[train_idx].nunique() == 2 and y.iloc[test_idx].nunique() == 2:
            return train_idx, test_idx, seed
    raise RuntimeError("Unable to create a subject-aware split containing both classes.")

def fit_prevalence_filter(X_train_raw, min_prevalence=0.05):
    prevalence = (X_train_raw > 0).mean(axis=0)
    selected_features = prevalence[prevalence >= min_prevalence].index.tolist()
    if not selected_features:
        raise ValueError("Prevalence filter removed every feature. Lower min_prevalence.")
    return selected_features

metadata = load_hmp2_metadata(metadata_path)
X_genus = load_genus_abundance_matrix(taxonomy_path)
data = merge_metadata_and_features(metadata, X_genus)

feature_columns = X_genus.columns.tolist()
y = data["IBD_Status"].astype(int)
groups = data["Participant ID"].astype(str)
X_raw = data[feature_columns]

train_idx, test_idx, split_seed = subject_aware_train_test_split(X_raw, y, groups, random_state=RANDOM_STATE)

X_train_raw = X_raw.iloc[train_idx].copy()
X_test_raw = X_raw.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()
groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

selected_features = fit_prevalence_filter(X_train_raw, min_prevalence=0.05)

X_train = np.log1p(X_train_raw[selected_features])
X_test = np.log1p(X_test_raw[selected_features])
X_all_model = np.log1p(X_raw[selected_features])

print("Merged modeling table:", data.shape)
print("Raw genus features:", len(feature_columns))
print("Selected genus features after training-only prevalence filter:", len(selected_features))
print("Split seed:", split_seed)
print("Train samples:", len(train_idx), "| Test samples:", len(test_idx))
print("Train participants:", groups_train.nunique(), "| Test participants:", groups_test.nunique())
print("Subject overlap between train and test:", len(set(groups_train).intersection(set(groups_test))))
print("\nSample-level diagnosis counts:")
display(data["diagnosis"].value_counts().to_frame("samples"))
print("\nSubject-level binary counts:")
display(data.groupby("Participant ID")["IBD_Status"].first().value_counts().rename(index={0: "nonIBD", 1: "IBD"}).to_frame("subjects"))

## 4. Pipeline Test Cases

These checks make the notebook more defensible and help prevent silent data mistakes.

In [ ]:
#@title 4.1 Validation Checks
test_results = []

def record_check(name, condition):
    passed = bool(condition)
    test_results.append({"check": name, "passed": passed})
    print(("PASS" if passed else "FAIL") + f": {name}")

record_check("1. Metadata file exists", metadata_path.exists() and metadata_path.stat().st_size > 0)
record_check("2. Taxonomy file exists", taxonomy_path.exists() and taxonomy_path.stat().st_size > 0)
record_check("3. Metadata loaded with rows", len(metadata) > 0)
record_check("4. Genus feature matrix loaded with rows and columns", X_genus.shape[0] > 0 and X_genus.shape[1] > 0)
record_check("5. Sample IDs successfully matched between metadata and taxonomy", len(data) > 0)
record_check("6. Target has exactly two classes for binary classification", y.nunique() == 2)
record_check("7. Both classes appear in train and test sets", y_train.nunique() == 2 and y_test.nunique() == 2)
record_check("8. No subject appears in both train and test", len(set(groups_train).intersection(set(groups_test))) == 0)
record_check("9. Feature matrix is numeric", all(pd.api.types.is_numeric_dtype(X_train_raw[c]) for c in selected_features))
record_check("10. No NaN remains after preprocessing", not X_train.isna().any().any() and not X_test.isna().any().any())
record_check("11. No infinite values remain after preprocessing", np.isfinite(X_train.to_numpy()).all() and np.isfinite(X_test.to_numpy()).all())
record_check("12. Prevalence filtering leaves at least 10 features", len(selected_features) >= 10)
record_check("13. Train/test row counts match labels", len(X_train) == len(y_train) and len(X_test) == len(y_test))
record_check("14. Binary class values are only 0 and 1", set(y.unique()) == {0, 1})
record_check("15. Output directory is writable", OUTPUT_DIR.exists())

test_summary = pd.DataFrame(test_results)
display(test_summary)

if not test_summary["passed"].all():
    raise AssertionError("One or more validation checks failed. Fix before modeling.")

## 5. Exploratory Data Analysis

The EDA focuses on the points most relevant to microbiome ML:

- Class balance.
- Sparsity and high dimensionality.
- PCA visualization of transformed genus-level abundance profiles.

In [ ]:
#@title 5.1 EDA plots
class_counts = data["diagnosis"].value_counts()
binary_counts = data["IBD_Status"].map({0: "nonIBD", 1: "IBD"}).value_counts()
sparsity = (X_raw[selected_features] == 0).mean().mean()

print(f"Average zero fraction across selected features: {sparsity:.2%}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, ax=axes[0], palette="Set2")
axes[0].set_title("Sample Counts by Diagnosis")
axes[0].set_xlabel("Diagnosis")
axes[0].set_ylabel("Samples")

sns.barplot(x=binary_counts.index, y=binary_counts.values, ax=axes[1], palette="Set1")
axes[1].set_title("Binary Target Counts")
axes[1].set_xlabel("Target")
axes[1].set_ylabel("Samples")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "class_balance.png", dpi=180, bbox_inches="tight")
plt.show()

pca_input = StandardScaler().fit_transform(X_all_model)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca_coords = pca.fit_transform(pca_input)
pca_df = pd.DataFrame({
    "PC1": pca_coords[:, 0],
    "PC2": pca_coords[:, 1],
    "Diagnosis": data["diagnosis"].values,
    "IBD Status": data["IBD_Status"].map({0: "nonIBD", 1: "IBD"}).values,
})

plt.figure(figsize=(7, 5))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="Diagnosis", alpha=0.75, s=35)
plt.title("PCA of Genus-Level Microbiome Abundance")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.legend(title="Diagnosis", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pca_genus_plot.png", dpi=180, bbox_inches="tight")
plt.show()

## 6. Supervised Model Comparison

The model comparison includes a majority-class baseline so accuracy can be interpreted correctly. Because the IBD class is the majority class in this merged sample-level dataset, a naive model can look good on raw accuracy while having zero specificity.

In [ ]:
#@title 6.1 Train and compare supervised models
def evaluate_predictions(model_name, y_true, y_pred, y_score):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_score),
        "pr_auc": average_precision_score(y_true, y_score),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }

model_definitions = {
    "Majority baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=5000, class_weight="balanced", C=0.1, random_state=RANDOM_STATE)),
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=800,
        random_state=RANDOM_STATE,
        class_weight={0: 3, 1: 1},
        min_samples_leaf=2,
        n_jobs=-1,
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=800,
        random_state=RANDOM_STATE,
        class_weight={0: 3, 1: 1},
        min_samples_leaf=2,
        n_jobs=-1,
    ),
    "SVM (RBF)": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=7)),
    ]),
}

trained_models = {}
model_rows = []

for model_name, model in model_definitions.items():
    fitted_model = clone(model)
    fitted_model.fit(X_train, y_train)
    trained_models[model_name] = fitted_model

    y_pred = fitted_model.predict(X_test)
    if hasattr(fitted_model, "predict_proba"):
        y_score = fitted_model.predict_proba(X_test)[:, 1]
    elif hasattr(fitted_model, "decision_function"):
        y_score = fitted_model.decision_function(X_test)
    else:
        y_score = y_pred

    model_rows.append(evaluate_predictions(model_name, y_test, y_pred, y_score))

model_results = pd.DataFrame(model_rows).sort_values(["roc_auc", "balanced_accuracy"], ascending=False)
display(model_results.round(3))

model_results.to_csv(OUTPUT_DIR / "model_comparison_metrics.csv", index=False)
print("Saved model comparison:", OUTPUT_DIR / "model_comparison_metrics.csv")

## 7. Threshold Tuning for the Best Model

The best model is selected by ROC-AUC, then its probability threshold is tuned on a validation split from the training set only. This avoids choosing the threshold directly from the test set.

In [ ]:
#@title 7.1 Threshold tuning and final test evaluation
candidate_results = model_results[model_results["model"] != "Majority baseline"].copy()
best_model_name = candidate_results.iloc[0]["model"]
print("Best model by ROC-AUC:", best_model_name)

fit_idx, val_idx = next(
    GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=7).split(X_train, y_train, groups_train)
)

threshold_model = clone(model_definitions[best_model_name])
threshold_model.fit(X_train.iloc[fit_idx], y_train.iloc[fit_idx])

if hasattr(threshold_model, "predict_proba"):
    val_score = threshold_model.predict_proba(X_train.iloc[val_idx])[:, 1]
else:
    val_score = threshold_model.decision_function(X_train.iloc[val_idx])

threshold_rows = []
for threshold in np.linspace(0.05, 0.95, 181):
    val_pred = (val_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_train.iloc[val_idx], val_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else 0
    recall_value = recall_score(y_train.iloc[val_idx], val_pred, zero_division=0)
    threshold_rows.append({
        "threshold": threshold,
        "balanced_accuracy": balanced_accuracy_score(y_train.iloc[val_idx], val_pred),
        "f1": f1_score(y_train.iloc[val_idx], val_pred, zero_division=0),
        "recall_sensitivity": recall_value,
        "specificity": specificity,
    })

threshold_table = pd.DataFrame(threshold_rows)
eligible_thresholds = threshold_table[threshold_table["recall_sensitivity"] >= 0.60].copy()
if eligible_thresholds.empty:
    selected_threshold = threshold_table.sort_values(["balanced_accuracy", "f1"], ascending=False).iloc[0]["threshold"]
else:
    selected_threshold = eligible_thresholds.sort_values(["balanced_accuracy", "f1"], ascending=False).iloc[0]["threshold"]

print(f"Selected threshold from validation data: {selected_threshold:.3f}")
display(threshold_table.sort_values(["balanced_accuracy", "f1"], ascending=False).head(10).round(3))

final_model = clone(model_definitions[best_model_name])
final_model.fit(X_train, y_train)

if hasattr(final_model, "predict_proba"):
    final_score = final_model.predict_proba(X_test)[:, 1]
else:
    final_score = final_model.decision_function(X_test)

final_pred_default = trained_models[best_model_name].predict(X_test)
final_pred_tuned = (final_score >= selected_threshold).astype(int)

final_default_metrics = evaluate_predictions(best_model_name + " default threshold", y_test, final_pred_default, final_score)
final_tuned_metrics = evaluate_predictions(best_model_name + " tuned threshold", y_test, final_pred_tuned, final_score)

final_metrics = pd.DataFrame([final_default_metrics, final_tuned_metrics])
display(final_metrics.round(3))
final_metrics.to_csv(OUTPUT_DIR / "final_model_metrics.csv", index=False)

print("\nObservation:")
print("- The majority baseline is useful only as a warning: high raw accuracy can hide poor healthy-control specificity.")
print("- The tuned threshold trades some IBD recall for better specificity, which makes the final model more clinically interpretable.")

## 8. Final Model Visualizations

The confusion matrix, ROC curve, and precision-recall curve are the most important figures for the PDF and presentation.

In [ ]:
#@title 8.1 Confusion matrix, ROC curve, and precision-recall curve
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    final_pred_tuned,
    display_labels=["nonIBD", "IBD"],
    cmap="Blues",
    ax=ax,
    colorbar=False,
)
ax.set_title(f"{best_model_name} Confusion Matrix\nTuned threshold = {selected_threshold:.2f}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix_final_model.png", dpi=180, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
RocCurveDisplay.from_predictions(y_test, final_score, ax=axes[0], name=best_model_name)
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
axes[0].set_title("ROC Curve")

PrecisionRecallDisplay.from_predictions(y_test, final_score, ax=axes[1], name=best_model_name)
axes[1].set_title("Precision-Recall Curve")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_pr_curves_final_model.png", dpi=180, bbox_inches="tight")
plt.show()

## 9. Feature Importance and Biological Interpretation

For tree-based models, feature importance gives a practical way to identify which genera contributed most to the prediction. These are not causal claims; they are model-derived signals that should be interpreted alongside microbiome literature.

In [ ]:
#@title 9.1 Top microbial genera
def extract_feature_importance(model, feature_names):
    if hasattr(model, "feature_importances_"):
        values = model.feature_importances_
    elif isinstance(model, Pipeline):
        final_step = model.steps[-1][1]
        if hasattr(final_step, "coef_"):
            values = np.abs(final_step.coef_).ravel()
        elif hasattr(final_step, "feature_importances_"):
            values = final_step.feature_importances_
        else:
            return None
    else:
        return None
    return pd.DataFrame({"feature": feature_names, "importance": values}).sort_values("importance", ascending=False)

feature_importance = extract_feature_importance(final_model, selected_features)

if feature_importance is None:
    print(f"Feature importance is not directly available for {best_model_name}.")
else:
    top_features = feature_importance.head(15).copy()
    display(top_features)
    feature_importance.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)

    plt.figure(figsize=(9, 6))
    plot_df = top_features.iloc[::-1].copy()
    sns.barplot(data=plot_df, x="importance", y="feature", palette="viridis")
    plt.title(f"Top Genus-Level Features: {best_model_name}")
    plt.xlabel("Model Importance")
    plt.ylabel("Taxonomic Feature")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "top_feature_importance.png", dpi=180, bbox_inches="tight")
    plt.show()

## 11. Literature Review Anchors

Use these sources in the written report and final presentation:

- **Lloyd-Price et al. (2019), Nature:** HMP2/IBDMDB followed IBD and non-IBD participants longitudinally and reported taxonomic, functional, biochemical, and host shifts in IBD.
- **Vangay et al. (2019), GigaScience:** MLRepo provides public benchmark microbiome machine learning tasks, including the Morgan 2012 IBD tasks used in milestone 1.
- **Manandhar et al. (2021):** supervised ML on fecal gut microbiome data showed predictive potential for IBD/non-IBD and CD/UC classification.
- **Yerke et al. (2024), Microbiome:** normalization choices matter in microbiome ML; relative abundance and simpler transformations can be competitive.
- **Quinn et al. (2019), GigaScience:** microbiome data are compositional, so taxonomic feature interpretation must be cautious.

## Final Inference

The final result should be framed as a research screening pipeline, not a clinical diagnostic tool. The model comparison tests whether microbial taxonomic abundance contains predictive signal for IBD status. The subject-aware split is the key methodological improvement because it prevents repeated samples from the same participant from appearing in both train and test sets.